# Import Modules

In [1]:
import os
import numpy as np
import pandas as pd
import random
import pickle
from tqdm import tqdm
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, Add
from tensorflow.keras.optimizers import Adam 
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping


Sekarang kita harus mengatur direktori untuk menggunakan data

In [2]:
# ======= LOAD DATASET ======= #
BASE_DIR = '/kaggle/input/flickr30k'
WORKING_DIR = '/kaggle/working'

# Load caption data
df = pd.read_csv(os.path.join(BASE_DIR, 'captions.txt'))
df.columns = ['image', 'caption']
df = df.dropna(subset=['caption'])  # Hapus caption NaN

# Buat mapping image ke caption
mapping = {}
for img, caption in zip(df['image'], df['caption']):
    mapping.setdefault(img, []).append("startseq " + str(caption).lower() + " endseq")

# Split dataset
image_ids = list(mapping.keys())
random.shuffle(image_ids)
split = int(len(image_ids) * 0.80)
train = image_ids[:split]
val = image_ids[split:]
print(f"Total Images: {len(image_ids)} | Train: {len(train)} | Val: {len(val)}")

Total Images: 31783 | Train: 25426 | Val: 6357


# Extract Image Features
Kita harus memuat dan menyusun ulang model

ResNet-50 adalah Residual Network adalah salah satu varian ResNet yang memiliki 50 layer, jenis arsitektur jaringan saraf yang telah merevolusi pembelajaran mendalam dengan memungkinkan pelatihan jaringan yang jauh lebih dalam dengan akurasi yang lebih baik dari sebelumnya. Terobosan dalam pembelajaran mendalam ini pertama kali diperkenalkan pada tahun 2015 oleh Kaiming He, Xiangyu Zhang, Shaoqing Ren, dan Jian Sun, peneliti di Microsoft.

In [3]:
# ==========================
# 2️⃣ **LOAD ResNet50 & CEK PICKLE**
# ==========================
pkl_path = os.path.join(WORKING_DIR, "features.pkl")

# Jika fitur sudah diekstrak sebelumnya, load dari pickle
if os.path.exists(pkl_path):
    print("✅ features.pkl ditemukan. Memuat fitur tanpa ekstraksi ulang...")
    with open(pkl_path, "rb") as f:
        features = pickle.load(f)
    print(f"Total features loaded: {len(features)}")

# Jika fitur belum ada, lakukan ekstraksi ulang
else:
    print("🔥 features.pkl tidak ditemukan. Akan melakukan ekstraksi ulang...")

    # Load ResNet50 Model dengan Global Average Pooling
    model_resnet = ResNet50(weights='imagenet', include_top=False, pooling='avg')


🔥 features.pkl tidak ditemukan. Akan melakukan ekstraksi ulang...
94773248/94765736 [==============================] - 1s 0us/step: 


Lapisan model ResNet50 yang terhubung sepenuhnya tidak diperlukan, hanya lapisan sebelumnya untuk mengekstrak hasil fitur.

Sesuai keinginan, Anda dapat menyertakan lebih banyak lapisan, tetapi untuk hasil yang lebih cepat, hindari menambahkan lapisan yang tidak perlu.

## extract the image features¶
Sekarang kita mengekstrak fitur gambar dan memuat data untuk praproses



In [ ]:
# ==========================
    # 3️⃣ **EKSTRAKSI FITUR DENGAN ResNet50**
    # ==========================
features = {}
print(f"Extracting features from {len(train) + len(val)} images...")
for img_name in tqdm(train + val, desc="Processing Images", unit="img"):
    img_path = os.path.join(BASE_DIR, 'Images', img_name)
    if os.path.exists(img_path):
        image = load_img(img_path, target_size=(224, 224))
        image = img_to_array(image)
        image = preprocess_input(np.expand_dims(image, axis=0))
        feature_vector = model_resnet.predict(image, verbose=0)

        image_id = img_name.split('.')[0]  # 🔥 Hilangkan ekstensi ".jpg"
        features[image_id] = feature_vector  # Simpan tanpa ".jpg"

print(f"✅ Extracted Features: {len(features)} images")

Processing Images:   0%|          | 0/31783 [00:00<?, ?img/s]

Extracting features from 31783 images...


Processing Images:  25%|██▍       | 7928/31783 [07:42<22:59, 17.29img/s] 

Kamus 'features' dibuat dan akan dimuat dengan fitur-fitur yang diekstrak dari data gambar

load_img(img_path, target_size=(224, 224)) - dimensi khusus untuk mengubah ukuran gambar saat dimuat ke dalam array

image.reshape((1, image.shape[0], image.shape[1], image.shape[2])) - membentuk ulang data gambar untuk diproses terlebih dahulu dalam gambar bertipe RGB.

model.predict(image, verbose=0) - ekstraksi fitur dari gambar

img_name.split('.')[0] - pemisahan nama gambar dari ekstensi untuk memuat hanya nama gambar.

In [ ]:
  # ==========================
    # 4️⃣ **SIMPAN FITUR KE PICKLE**
    # ==========================
with open(pkl_path, "wb") as f:
    pickle.dump(features, f)
    print("✅ Fitur berhasil diekstrak dan disimpan di features.pkl")

Fitur yang diekstrak tidak disimpan dalam disk, jadi ekstraksi ulang fitur dapat memperpanjang waktu berjalan

Membuang dan menyimpan kamus Anda dalam pickle untuk memuat ulang guna menghemat waktu

In [ ]:

# Load features from pickle
with open(os.path.join(WORKING_DIR, 'features.pkl'), 'rb') as f:
    features = pickle.load(f)

Muat semua data fitur yang tersimpan ke proyek Anda untuk waktu proses yang lebih cepat


## Load the Captions Data
Mari kita simpan data teks dari file teks



In [ ]:
with open(os.path.join(BASE_DIR, 'captions.txt'), 'r') as f:
    next(f)
    captions_doc = f.read()

Sekarang kita pisahkan dan tambahkan data teks dengan gambar

In [ ]:
# Mapping image ke caption
mapping = {}
for line in tqdm(captions_doc.split('\n')):
    tokens = line.split(',')
    if len(line) < 2:
        continue
    image_id, caption = tokens[0], tokens[1:]
    image_id = image_id.split('.')[0]  # 🔥 Hilangkan ekstensi ".jpg"
    caption = " ".join(caption)
    if image_id not in mapping:
        mapping[image_id] = []
    mapping[image_id].append(caption)

library 'mapping' dibuat dengan kunci sebagai image_id dan nilai sebagai teks keterangan yang sesuai

Gambar yang sama dapat memiliki beberapa keterangan, jika image_id tidak ada dalam mapping: mapping[image_id] = [] membuat daftar untuk menambahkan keterangan ke gambar yang sesuai

**Sekarang mari kita lihat jumlah gambar yang diunggah**


In [ ]:
len(mapping)

## Preprocess Text Data

In [ ]:
# ==========================
# 6️⃣ **CLEANING TEXT DATA**
# ==========================
import re

def clean(mapping):
    for key, captions in mapping.items():
        for i in range(len(captions)):
            caption = captions[i].lower()
            caption = re.sub(r'[^A-Za-z ]', '', caption)
            caption = re.sub(r'\s+', ' ', caption).strip()
            words = caption.split()
            if words and words[0] == "startseq":
                words.pop(0)
            if words and words[-1] == "endseq":
                words.pop()
            words.insert(0, "startseq")
            words.append("endseq")
            captions[i] = " ".join(words)

print("✅ Before Cleaning:", mapping[list(mapping.keys())[0]])
clean(mapping)
print("✅ After Cleaning:", mapping[list(mapping.keys())[0]])

Ditetapkan untuk membersihkan dan mengubah teks untuk proses yang lebih cepat dan hasil yang lebih baik

Mari kita visualisasikan teks sebelum dan sesudah dibersihkan

In [ ]:
# before preprocess of text
mapping['1000344755']

In [ ]:
# preprocess the text
clean(mapping)

In [ ]:
# after preprocess of text
mapping['1000092795']

**Selanjutnya kita akan menyimpan teks yang telah diproses sebelumnya ke dalam sebuah daftar**

In [ ]:
all_captions = []
for key in mapping:
    for caption in mapping[key]:
        all_captions.append(caption)

In [ ]:
len(all_captions)

Jumlah teks unik yang disimpan

## 10 Captions
Mari kita lihat sepuluh teks pertama

In [ ]:
all_captions[:10]

# Processing of Text Data
Sekarang kita mulai memproses data teks

In [ ]:
# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index) + 1


In [ ]:
vocab_size


Jumlah kata unik



In [ ]:
# get maximum length of the caption available
max_length = max(len(caption.split()) for caption in all_captions)
max_length


Menemukan panjang maksimum teks, digunakan sebagai referensi untuk urutan padding.


# Train Valid Split

Setelah melakukan preprocessing data sekarang kita akan melakukan training, validasi dan split



In [ ]:
image_ids = list(mapping.keys())
split = int(len(image_ids) * 0.80)
train = image_ids[:split]
valid = image_ids[split:]

Sekarang kita akan mendefinisikan batch dan menyertakan urutan padding



In [ ]:
def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = [], [], []
    n = 0
    while True:
        for key in data_keys:
            captions = mapping[key]
            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]  # ✅ Label tetap dalam bentuk indeks integer
                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]

                    X1.append(features[key][0])
                    X2.append(in_seq)
                    y.append(out_seq)  # ✅ Label tetap integer, bukan one-hot encoding

            n += 1
            if n == batch_size:
                yield [np.array(X1), np.array(X2)], np.array(y)  # ✅ Pastikan y tetap integer
                X1, X2, y = [], [], []
                n = 0

In [ ]:
# ==========================
# 6️⃣ Model Image Captioning untuk ResNet50
# ==========================
inputs1 = Input(shape=(2048,))  # ResNet50 menghasilkan 2048 fitur
fe1 = Dropout(0.4)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)

# Sequence feature layers (Decoder)
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.4)(se1)
se3 = LSTM(256)(se2)

# Decoder model
decoder1 = Add()([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)


### Learning Rate

In [ ]:
from tensorflow.keras.utils import plot_model

# ==========================
# 7️⃣ Learning Rate & Callbacks
# ==========================
learning_rate = 0.0001
optimizer = Adam(learning_rate=learning_rate)

model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['sparse_categorical_accuracy'])

# Plot model
plot_model(model, show_shapes=True)

# Callbacks untuk overfitting
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1) 


In [ ]:
print("Total extracted features:", len(features))
print("Sample keys from features:", list(features.keys())[:5])  # Cek beberapa ID gambar

# Model Creation

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, Add
from tensorflow.keras.utils import plot_model
from tensorflow.keras.metrics import SparseCategoricalAccuracy

# ⚡ ResNet50 Feature Extractor (Output 2048-dimensi)
inputs1 = Input(shape=(2048,))
fe1 = Dropout(0.4)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)  # Ubah ke 256-dimensi

# ⚡ Caption Feature Extractor
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.4)(se1)
se3 = LSTM(256)(se2)

# ⚡ Decoder (Gabungkan fitur gambar & teks)
decoder1 = Add()([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

# Buat model
model = Model(inputs=[inputs1, inputs2], outputs=outputs)

# Compile model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=[SparseCategoricalAccuracy()])

# 🔥 Plot model
plot_model(model, show_shapes=True)

# Simpan model sebagai file PNG
PLOT_PATH = "resnet50_lstm_model.png"
plot_model(model, to_file=PLOT_PATH, show_shapes=True, show_layer_names=True)

## Train Model
Sekarang mari kita melatih modelnya



In [ ]:
print("Total extracted features:", len(features))
print("Sample keys from features:", list(features.keys())[:5])  # Cek beberapa ID gambar


In [ ]:
# ==========================
# 8️⃣ Training Model untuk ResNet50
# ==========================
train_losses, valid_losses = [], []
train_accuracies, valid_accuracies = [], []

epochs = 30
batch_size = 32
steps_train = len(train) // batch_size
steps_valid = len(valid) // batch_size

for i in range(epochs):
    print(f"\n **Epoch {i+1}/{epochs}** ")

    train_generator = data_generator(train, mapping, features, tokenizer, max_length, vocab_size, batch_size)
    valid_generator = data_generator(valid, mapping, features, tokenizer, max_length, vocab_size, batch_size)

    history_train = model.fit(
        train_generator,
        epochs=1,
        steps_per_epoch=steps_train,
        verbose=1,
        callbacks=[reduce_lr, early_stop]
    )

    history_valid = model.evaluate(valid_generator, steps=steps_valid, verbose=1)

    train_loss = history_train.history['loss'][0]
    train_accuracy = history_train.history['sparse_categorical_accuracy'][0]
    valid_loss = history_valid[0]
    valid_accuracy = history_valid[1]

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    valid_losses.append(valid_loss)
    valid_accuracies.append(valid_accuracy)

    print(f"🎯 Epoch {i+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f} | Valid Loss: {valid_loss:.4f} | Valid Accuracy: {valid_accuracy:.4f}")

    if early_stop.stopped_epoch > 0:
        print("\n⚠️ **Early Stopping Triggered! Training Stopped Early.** ⚠️")
        break

### save model

In [ ]:
MODEL_PATH = "image_captioning_resnet50_lstm.h5"
model.save(MODEL_PATH)
print(f"✅ Model berhasil disimpan di: {MODEL_PATH}")


### save history training


In [ ]:
import json

# Simpan history training ke JSON
history_data = {
    "loss": train_losses,
    "val_loss": valid_losses,
    "sparse_categorical_accuracy": train_accuracies,
    "val_sparse_categorical_accuracy": valid_accuracies
}

HISTORY_PATH = "resnet50_lstm_training_history.json"

with open(HISTORY_PATH, "w") as f:
    json.dump(history_data, f)

print(f"✅ History training berhasil disimpan di: {HISTORY_PATH}")

### save tokenizer

In [ ]:
import pickle

# Simpan tokenizer ke file
TOKENIZER_PATH = "tokenizer.pkl"

with open(TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer, f)

print(f"✅ Tokenizer berhasil disimpan di: {TOKENIZER_PATH}")

### save mapping

In [ ]:
import pickle

# Simpan mapping caption ke file pickle
MAPPING_PATH = "mapping.pkl"
with open(MAPPING_PATH, "wb") as f:
    pickle.dump(mapping, f)

print(f"✅ Mapping caption berhasil disimpan di: {MAPPING_PATH}")

# Graph

In [ ]:
import matplotlib.pyplot as plt

# Load history dari JSON
with open(HISTORY_PATH, "r") as f:
    history_data = json.load(f)

train_losses = history_data["loss"]
val_losses = history_data["val_loss"]
train_accuracies = history_data["sparse_categorical_accuracy"]
val_accuracies = history_data["val_sparse_categorical_accuracy"]

# Plot Loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss", marker='o', color='red')
plt.plot(val_losses, label="Validation Loss", marker='o', color='green')
plt.title("Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Train Accuracy", marker='o', color='blue')
plt.plot(val_accuracies, label="Validation Accuracy", marker='o', color='orange')
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()

plt.show()

# Load Model .h5 + + testing 8k + cosine + foldering

ekstraksi flickr 8k dulu

## extract test 8k

In [ ]:
import os
import numpy as np
import pickle
from tqdm import tqdm
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model

# 🔹 Load ResNet50 Model untuk Ekstraksi Fitur
base_model = ResNet50(weights="imagenet")
model_resnet = Model(inputs=base_model.input, outputs=base_model.layers[-2].output)

# 🔹 Path ke Dataset Flickr8k
FLICKR8K_PATH = "/kaggle/input/flickr8k/Images"  # Sesuaikan dengan lokasi dataset Flickr8k

# 🔹 Ekstraksi Fitur Gambar dari Flickr8k
features_8k = {}
image_files = [f for f in os.listdir(FLICKR8K_PATH) if f.endswith(".jpg")]

print(f"🔄 Memproses {len(image_files)} gambar dari Flickr8k...")

for img_name in tqdm(image_files):
    img_path = os.path.join(FLICKR8K_PATH, img_name)

    # Load & Preprocess Gambar
    image = load_img(img_path, target_size=(224, 224))
    image_array = img_to_array(image)
    image_array = np.expand_dims(image_array, axis=0)
    image_array = preprocess_input(image_array)

    # Ekstrak fitur dengan ResNet50
    feature = model_resnet.predict(image_array, verbose=0)

    # Simpan fitur dengan ID gambar (tanpa ekstensi .jpg)
    image_id = os.path.splitext(img_name)[0]
    features_8k[image_id] = feature

# 🔹 Simpan Fitur Gambar Flickr8k dalam Pickle
FEATURES_8K_PATH = "features_8k.pkl"

with open(FEATURES_8K_PATH, "wb") as f:
    pickle.dump(features_8k, f)

print(f"✅ Fitur gambar Flickr8k berhasil diekstrak dan disimpan di: {FEATURES_8K_PATH}")


## Load Model, Tokenizer, Feature (8k), Mapping

In [ ]:
from tensorflow.keras.models import load_model
import pickle
import os
import random

# Path ke dataset Flickr8k
FLICKR8K_PATH = "/kaggle/input/flickr8k/Images"  # Sesuaikan jika berbeda

# Load model yang telah dilatih
MODEL_PATH = "/kaggle/input/dataset-epoch30/image_captioning_resnet50_lstm.h5"
model = load_model(MODEL_PATH)
print("✅ Model berhasil dimuat!")

# Load tokenizer yang digunakan saat training
TOKENIZER_PATH = "/kaggle/input/dataset-epoch30/tokenizer.pkl"
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)
print("✅ Tokenizer berhasil dimuat!")

# Load fitur gambar yang sudah diekstrak
FEATURES_PATH = "/kaggle/input/dataset-epoch30/features_8k.pkl"  # Gunakan fitur dari Flickr8k
with open(FEATURES_PATH, "rb") as f:
    features = pickle.load(f)
print("✅ Fitur gambar Flickr8k berhasil dimuat!")

# Load mapping image-caption
MAPPING_PATH = "/kaggle/input/dataset-epoch30/mapping.pkl"
with open(MAPPING_PATH, "rb") as f:
    mapping = pickle.load(f)
print("✅ Mapping caption berhasil dimuat!")


## mapping captionnya

In [ ]:
import os
import pandas as pd

# Path ke dataset captions Flickr8k
CAPTIONS_PATH = "/kaggle/input/flickr8k/captions.txt"

# Load file captions.txt ke dalam DataFrame
df = pd.read_csv(CAPTIONS_PATH)
df.columns = ["image", "caption"]

# Buat mapping image_id -> daftar caption
mapping = {}
for img, caption in zip(df["image"], df["caption"]):
    image_id = img.split(".")[0]  # Hilangkan ekstensi .jpg
    mapping.setdefault(image_id, []).append(caption.lower())  # Simpan caption dalam lowercase

print(f"✅ Mapping captions berhasil dimuat! Total gambar: {len(mapping)}")


### cek gambar yang dipilih (random) 8k

In [ ]:
# Ambil semua file gambar dari dataset Flickr8k
image_files = [f for f in os.listdir(FLICKR8K_PATH) if f.endswith(".jpg")]

# Pilih satu gambar secara random
random_image = random.choice(image_files)
print(f"🎯 Gambar yang dipilih: {random_image}")


## Model Validation

### Cosine Similarity

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Fungsi untuk mengonversi indeks ke kata dalam tokenizer
def idx_to_word(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None

# Fungsi untuk memprediksi caption dari gambar
def predict_caption(model, image, tokenizer, max_length):
    in_text = 'startseq'
    
    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], max_length)
        
        # Prediksi kata berikutnya
        yhat = model.predict([image, sequence], verbose=0)
        yhat = np.argmax(yhat)  # Ambil indeks kata dengan probabilitas tertinggi
        
        # Konversi indeks ke kata
        word = idx_to_word(yhat, tokenizer)
        
        # Jika kata tidak ditemukan, stop prediksi
        if word is None:
            break
        
        # Tambahkan kata ke caption
        in_text += " " + word
        
        # Stop jika sudah mencapai akhir sequence
        if word == 'endseq':
            break
    
    # Hapus "startseq" dan "endseq" dari caption
    return in_text.replace('startseq', '').replace('endseq', '').strip()


In [ ]:
# Hitung panjang maksimal caption dari tokenizer
max_length = max(len(caption.split()) for captions in mapping.values() for caption in captions)

print(f"✅ `max_length` berhasil dihitung: {max_length}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(pred_caption, actual_captions):
    """Menghitung Cosine Similarity antara caption prediksi dan caption asli."""
    if actual_captions == ["Tidak ada caption asli"]:
        return 0.0  # Jika tidak ada caption asli, langsung kembalikan skor 0.0
    
    vectorizer = TfidfVectorizer()
    text_data = actual_captions + [pred_caption]  # Gabungkan semua caption

    # Transform teks menjadi vektor TF-IDF
    tfidf_matrix = vectorizer.fit_transform(text_data)

    # Ambil vektor caption prediksi
    pred_vector = tfidf_matrix[-1]

    # Hitung Cosine Similarity antara prediksi dan setiap caption asli
    similarities = cosine_similarity(pred_vector, tfidf_matrix[:-1]).flatten()

    # Ambil nilai Cosine Similarity tertinggi sebagai representasi terbaik
    return np.max(similarities)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(pred_caption, actual_captions):
    """Menghitung Cosine Similarity antara caption prediksi dan caption asli."""
    vectorizer = TfidfVectorizer()
    text_data = actual_captions + [pred_caption]  # Gabungkan semua caption

    # Transform teks menjadi vektor TF-IDF
    tfidf_matrix = vectorizer.fit_transform(text_data)

    # Ambil vektor caption prediksi
    pred_vector = tfidf_matrix[-1]

    # Hitung Cosine Similarity antara prediksi dan setiap caption asli
    similarities = cosine_similarity(pred_vector, tfidf_matrix[:-1]).flatten()

    # Ambil nilai Cosine Similarity tertinggi sebagai representasi terbaik
    return np.max(similarities)


In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import os
import shutil

# Ambil semua file gambar dari dataset Flickr8k
image_files = [f for f in os.listdir(FLICKR8K_PATH) if f.endswith(".jpg")]

# **Prioritas kategori dari yang paling penting ke yang paling umum**
# category_priority = ["pemandangan", "hewan", "kegiatan", "manusia", "suasana"]
category_priority = ["pemandangan", "hewan", "kegiatan", "manusia"]

# **Kata kunci untuk kategori caption**
category_keywords = {
    "pemandangan": ["view", "mountain", "sea", "sunset", "nature", "landscape", "ocean", "scenery", "river", "beach", "shore", "lake"],
    "hewan": ["animal", "dog", "cat", "bird", "fish", "horse", "elephant", "tiger", "lion"],
    "kegiatan": ["walking", "running", "jumping", "riding", "swimming", "cycling", "dancing", "playing"],
    "manusia": ["human", "person", "man", "woman", "people", "child", "boy", "girl", "adult"],
    # "suasana": ["calm", "quiet", "peaceful", "relax", "serene", "tranquil", "silence"]
}

# ✅ **Fungsi untuk memilih kategori dengan sistem prioritas**
def categorize_image_by_keywords(caption):
    words = set(caption.lower().split())
    matched_categories = []

    # Cek apakah caption mengandung kata kunci dari kategori yang ada
    for category in category_priority:
        keywords = category_keywords[category]
        if any(word in words for word in keywords):
            matched_categories.append(category)

    # Jika ada kategori yang cocok, pilih kategori dengan prioritas tertinggi
    if matched_categories:
        return matched_categories[0]  # Ambil kategori dengan prioritas tertinggi
    
    return "uncategorized"  # Jika tidak ada kecocokan, masukkan ke kategori "uncategorized"

# ✅ **Fungsi untuk menyimpan gambar ke folder kategori**
def save_image_to_category_folder(image_name, category, base_path="categorized_images"):
    folder_path = os.path.join(base_path, category)
    os.makedirs(folder_path, exist_ok=True)
    
    img_path = os.path.join(FLICKR8K_PATH, image_name)
    shutil.copy(img_path, os.path.join(folder_path, image_name))  
    print(f"✅ Gambar {image_name} disimpan ke kategori: {category}")

# ✅ **Fungsi untuk menghitung Cosine Similarity**
def calculate_cosine_similarity(pred_caption, actual_captions):
    if actual_captions == ["Tidak ada caption asli"]:
        return 0.0  

    vectorizer = TfidfVectorizer()
    text_data = actual_captions + [pred_caption]  
    tfidf_matrix = vectorizer.fit_transform(text_data)
    pred_vector = tfidf_matrix[-1]

    similarities = cosine_similarity(pred_vector, tfidf_matrix[:-1]).flatten()
    return np.max(similarities)

# ✅ **Fungsi untuk mengonversi indeks menjadi kata dalam tokenizer**
def idx_to_word(integer, tokenizer):
    """Mengonversi indeks integer menjadi kata berdasarkan tokenizer."""
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None  # Jika indeks tidak ditemukan, kembalikan None

# ✅ **Fungsi untuk prediksi caption (Pastikan padding sesuai `max_length`)**
def predict_caption(model, image_feature, tokenizer, max_length):
    in_text = 'startseq'
    
    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)  

        yhat = model.predict([image_feature, sequence], verbose=0)
        yhat = np.argmax(yhat)  

        word = idx_to_word(yhat, tokenizer)
        if word is None:
            break

        in_text += " " + word
        if word == 'endseq':
            break
    
    return in_text.replace('startseq', '').replace('endseq', '').strip()

# ✅ **Fungsi utama untuk captioning & Cosine Similarity**
def generate_caption_with_cosine():
    random_image = random.choice(image_files)
    image_id = os.path.splitext(random_image)[0]  
    img_path = os.path.join(FLICKR8K_PATH, random_image)

    image = Image.open(img_path)
    actual_captions = mapping.get(image_id, ["Tidak ada caption asli"])

    # **Ambil fitur gambar dari features.pkl**
    image_feature = features.get(image_id)
    if image_feature is None:
        print(f"❌ Fitur gambar untuk {image_id} tidak ditemukan!")
        return
    
    # **Prediksi caption**
    y_pred = predict_caption(model, image_feature, tokenizer, max_length)
    similarity_score = calculate_cosine_similarity(y_pred, actual_captions)
    
    # **Gunakan kategori dengan sistem prioritas**
    category = categorize_image_by_keywords(y_pred)
    
    # **Simpan gambar ke dalam folder kategori**
    save_image_to_category_folder(random_image, category)

    # print("\n--------------------- Actual Captions ---------------------")
    # for caption in actual_captions:
    #     print(caption)

    # print("\n--------------------- Predicted Caption ---------------------")
    # print(y_pred)

    print(f"📂 **Kategori:** {category}")

    plt.imshow(image)
    plt.axis("off")
    # plt.title(f"Predicted Caption: {y_pred}\nKategori: {category}\nCosine Similarity: {similarity_score:.4f}", fontsize=14)
    plt.title(f"Predicted Caption: {y_pred}\nKategori: {category}", fontsize=14)
    plt.show()

# 🔥 **Jalankan captioning dengan Cosine Similarity**
generate_caption_with_cosine()


cosine load 10 random image

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

# Ambil 10 gambar random
random_images = random.sample(image_files, 10)

# Loop untuk memproses 10 gambar
for random_image in random_images:
    image_id = os.path.splitext(random_image)[0]  
    img_path = os.path.join(FLICKR8K_PATH, random_image)

    # Load gambar
    image = Image.open(img_path)

    # Ambil caption asli dari dataset Flickr8k
    actual_captions = mapping.get(image_id, ["Tidak ada caption asli"])

    # Prediksi caption
    y_pred = predict_caption(model, features[image_id], tokenizer, max_length)

    # Hitung Cosine Similarity
    similarity_score = calculate_cosine_similarity(y_pred, actual_captions)

    # Tentukan kategori berdasarkan caption prediksi
    category = categorize_image_by_keywords(y_pred)

    # Simpan gambar ke dalam folder kategori
    save_image_to_category_folder(random_image, category)

    # Tampilkan hasil
    # print("\n--------------------- Actual Captions ---------------------")
    # for caption in actual_captions:
    #     print(caption)

    # print("\n--------------------- Predicted Caption ---------------------")
    # print(y_pred)

    # print(f"\n🔍 **Cosine Similarity:** {similarity_score:.4f}")
    print(f"📂 **Kategori:** {category}")

    # Tampilkan gambar
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    # plt.title(f"Predicted Caption: {y_pred}\nKategori: {category}\nCosine Similarity: {similarity_score:.4f}", fontsize=12)
    plt.title(f"Predicted Caption: {y_pred}\nKategori: {category}", fontsize=14)
    plt.show()


### Bleu-1 - Bleu-4

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

In [ ]:
from nltk.translate.bleu_score import SmoothingFunction

smooth = SmoothingFunction().method1

def calculate_bleu_score(actual_captions, predicted_caption):
    references = [caption.split() for caption in actual_captions]
    hypothesis = predicted_caption.split()

    if len(references) == 0 or len(hypothesis) == 0:
        return 0.0, 0.0, 0.0, 0.0

    bleu1 = corpus_bleu([references], [hypothesis], weights=(1.0, 0, 0, 0), smoothing_function=smooth)

    return bleu1


In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import os
import shutil
from nltk.translate.bleu_score import SmoothingFunction

# Ambil semua file gambar dari dataset Flickr8k
image_files = [f for f in os.listdir(FLICKR8K_PATH) if f.endswith(".jpg")]

smooth = SmoothingFunction().method1

# Fungsi utama untuk captioning, BLEU Score, & folderisasi
def generate_caption_with_bleu():
    # Pilih satu gambar secara random
    random_image = random.choice(image_files)
    image_id = os.path.splitext(random_image)[0]  # Ambil ID gambar tanpa ekstensi
    img_path = os.path.join(FLICKR8K_PATH, random_image)

    # Load gambar
    image = Image.open(img_path)

    # Ambil caption asli dari dataset Flickr8k
    actual_captions = mapping.get(image_id, ["Tidak ada caption asli"])

    # Prediksi caption
    y_pred = predict_caption(model, features[image_id], tokenizer, max_length)

    # Hitung BLEU Score
    # bleu1, bleu2, bleu3, bleu4 = calculate_bleu_score(actual_captions, y_pred)
    bleu1 = calculate_bleu_score(actual_captions, y_pred)

    # Tentukan kategori berdasarkan caption prediksi
    category = categorize_image_by_keywords(y_pred)

    # Simpan gambar ke dalam folder kategori
    save_image_to_category_folder(random_image, category)

    # Tampilkan hasil
    #  print("\n--------------------- Actual Captions ---------------------")
    # for caption in actual_captions:
    #     print(caption)

    # print("\n--------------------- Predicted Caption ---------------------")
    # print(y_pred)

    # print(f"📊 **BLEU-1 Score:** {bleu1:.4f}")
    print(f"📂 **Kategori:** {category}")

    # Tampilkan gambar
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Predicted Caption: {y_pred}\nKategori: {category}\nBLEU-1: {bleu1:.4f}", fontsize=14)
    plt.show()

# 🔥 Jalankan evaluasi BLEU Score & folderisasi
generate_caption_with_bleu()

In [ ]:
# Hapus jika perlu
import shutil

# Pastikan penyimpanan di directory yang bisa ditulis
CATEGORIZED_IMAGES_PATH = "/kaggle/working/categorized_images"

# Hapus folder jika sudah ada sebelumnya
if os.path.exists(CATEGORIZED_IMAGES_PATH):
    shutil.rmtree(CATEGORIZED_IMAGES_PATH)
    print("✅ Folder 'categorized_images' berhasil dihapus.")

# Buat ulang folder untuk menyimpan hasil baru
os.makedirs(CATEGORIZED_IMAGES_PATH, exist_ok=True)
print("✅ Folder 'categorized_images' dibuat ulang.")


In [ ]:
import random
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pastikan penyimpanan di directory yang bisa ditulis
CATEGORIZED_IMAGES_PATH = "/kaggle/working/categorized_images"

# Hapus folder jika sudah ada sebelumnya
if os.path.exists(CATEGORIZED_IMAGES_PATH):
    shutil.rmtree(CATEGORIZED_IMAGES_PATH)
    print("✅ Folder 'categorized_images' berhasil dihapus.")

# Buat ulang folder untuk menyimpan hasil baru
os.makedirs(CATEGORIZED_IMAGES_PATH, exist_ok=True)
print("✅ Folder 'categorized_images' dibuat ulang.")

# Pastikan jumlah sampel yang diambil
NUM_SAMPLES = 1618

# Ambil 1000 gambar secara acak dari dataset Flickr8k
random_images = random.sample(image_files, min(NUM_SAMPLES, len(image_files)))

# Simpan skor BLEU dan kategori
bleu_scores = []
category_counts = {key: 0 for key in category_priority}
category_counts["uncategorized"] = 0  # Tambahkan kategori uncategorized

# Fungsi untuk menyimpan gambar ke folder kategori

def save_image_to_category_folder(image_name, img_path, category, base_path=CATEGORIZED_IMAGES_PATH):
    folder_path = os.path.join(base_path, category)
    os.makedirs(folder_path, exist_ok=True)  # Pastikan folder kategori sudah ada

    dest_path = os.path.join(folder_path, image_name)

    if not os.path.exists(dest_path):
        shutil.copy2(img_path, dest_path)  # Salin gambar ke folder kategori

# Progress bar
for idx, random_image in enumerate(tqdm(random_images, desc="Processing Images", leave=True, dynamic_ncols=True)):
    image_id = os.path.splitext(random_image)[0]  # Ambil ID gambar tanpa ekstensi
    img_path = os.path.join(FLICKR8K_PATH, random_image)  # Pastikan hanya membaca, tidak menyimpan di input

    # Ambil caption asli dari dataset Flickr8k
    actual_captions = mapping.get(image_id, ["Tidak ada caption asli"])

    # Prediksi caption
    y_pred = predict_caption(model, features[image_id], tokenizer, max_length)

    # Hitung BLEU-1 Score
    bleu1 = calculate_bleu_score(actual_captions, y_pred)
    bleu_scores.append(bleu1)

    # Tentukan kategori berdasarkan caption prediksi
    category = categorize_image_by_keywords(y_pred)
    category_counts[category] += 1  # Tambahkan ke kategori yang sesuai

    # Simpan gambar ke dalam folder kategori
    save_image_to_category_folder(random_image, img_path, category)

# Hitung rata-rata BLEU-1
average_bleu1 = np.mean(bleu_scores)

# Tampilkan hasil akhir
print("\n=============================")
print(f"📊 Rata-rata BLEU-1 Score: {average_bleu1:.4f}")
print("📂 Distribusi kategori:")
for category, count in category_counts.items():
    print(f"  - {category}: {count} gambar")
print("=============================")

# Tampilkan distribusi kategori dalam bentuk chart
plt.figure(figsize=(10, 6))
plt.bar(category_counts.keys(), category_counts.values(), color="skyblue")
plt.xlabel("Kategori")
plt.ylabel("Jumlah Gambar")
plt.title("Distribusi Kategori dari 1618 Gambar Flickr8k")
plt.xticks(rotation=45)
plt.show()

bleu 10 random image

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import os
import shutil
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# Ambil semua file gambar dari dataset Flickr8k
image_files = [f for f in os.listdir(FLICKR8K_PATH) if f.endswith(".jpg")]

smooth = SmoothingFunction().method1

# ✅ Fungsi untuk menyimpan gambar ke folder kategori
def save_image_to_category_folder(image_name, img_path, category, base_path="categorized_images"):
    folder_path = os.path.join(base_path, category)
    os.makedirs(folder_path, exist_ok=True)  # Pastikan folder kategori sudah ada

    dest_path = os.path.join(folder_path, image_name)

    # Hanya salin jika file belum ada di tujuan
    if not os.path.exists(dest_path):
        shutil.copy2(img_path, dest_path)
        print(f"✅ Gambar {image_name} disimpan ke kategori: {category}")
    else:
        print(f"⚠️ Gambar {image_name} sudah ada di kategori {category}, tidak disalin lagi.")

# Fungsi untuk menghitung BLEU Score
def calculate_bleu_score(actual_captions, predicted_caption):
    references = [caption.split() for caption in actual_captions]
    hypothesis = predicted_caption.split()

    if len(references) == 0 or len(hypothesis) == 0:
        return 0.0, 0.0, 0.0, 0.0

    bleu1 = corpus_bleu([references], [hypothesis], weights=(1.0, 0, 0, 0), smoothing_function=smooth)
    # bleu2 = corpus_bleu([references], [hypothesis], weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
    # bleu3 = corpus_bleu([references], [hypothesis], weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
    # bleu4 = corpus_bleu([references], [hypothesis], weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)

    # return bleu1, bleu2, bleu3, bleu4
    return bleu1

# Ambil 10 gambar random
num_images = min(10, len(image_files))  # Jika kurang dari 10, ambil semua yang tersedia
random_images = random.sample(image_files, num_images)

# Loop untuk memproses 10 gambar
for random_image in random_images:
    image_id = os.path.splitext(random_image)[0]  
    img_path = os.path.join(FLICKR8K_PATH, random_image)

    # Load gambar
    image = Image.open(img_path)

    # Ambil caption asli dari dataset Flickr8k
    actual_captions = mapping.get(image_id, ["Tidak ada caption asli"])

    # Prediksi caption
    y_pred = predict_caption(model, features[image_id], tokenizer, max_length)

    # Hitung BLEU Score
    # bleu1, bleu2, bleu3, bleu4 = calculate_bleu_score(actual_captions, y_pred)
    bleu1 = calculate_bleu_score(actual_captions, y_pred)

    # Tentukan kategori berdasarkan caption prediksi
    category = categorize_image_by_keywords(y_pred)

    # Simpan gambar ke dalam folder kategori
    save_image_to_category_folder(random_image, img_path, category)

    # Tampilkan hasil
    #  print("\n--------------------- Actual Captions ---------------------")
    # for caption in actual_captions:
    #     print(caption)

    # print("\n--------------------- Predicted Caption ---------------------")
    # print(y_pred)

    print(f"📊 **BLEU-1 Score:** {bleu1:.4f}")
    # print(f"📊 **BLEU-2 Score:** {bleu2:.4f}")
    # print(f"📊 **BLEU-3 Score:** {bleu3:.4f}")
    # print(f"📊 **BLEU-4 Score:** {bleu4:.4f}")
    print(f"📂 **Kategori:** {category}")

    # Tampilkan gambar
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Predicted Caption: {y_pred}\nKategori: {category}\nBLEU-1: {bleu1:.4f}", fontsize=14)
    plt.show()


### Another

In [ ]:
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
import numpy as np

# Load model ResNet50 untuk ekstraksi fitur (tanpa fully connected layer terakhir)
base_model = ResNet50(weights="imagenet")
feature_extractor = Model(inputs=base_model.input, outputs=base_model.layers[-2].output)

def extract_features(image_path):
    """Ekstraksi fitur dari gambar yang diunggah menggunakan ResNet50."""
    image = load_img(image_path, target_size=(224, 224))  # Load gambar & ubah ukurannya ke 224x224
    image_array = img_to_array(image)  # Konversi ke array
    image_array = np.expand_dims(image_array, axis=0)  # Tambahkan batch dimension
    image_array = preprocess_input(image_array)  # Preprocessing untuk ResNet50

    # Ekstrak fitur menggunakan model ResNet50
    feature = feature_extractor.predict(image_array, verbose=0).reshape(1, 2048)
    return feature

print("✅ Fungsi `extract_features()` berhasil didefinisikan!")


In [ ]:
import os
import shutil

def save_image_to_category_folder(image_name, image_path, category, base_path="categorized_images"):
    """Simpan gambar ke dalam folder berdasarkan kategori caption."""
    folder_path = os.path.join(base_path, category)
    
    # Cek apakah folder sudah ada sebelum membuatnya
    if not os.path.exists(folder_path):
        os.makedirs(folder_path, exist_ok=True)
    
    # Path tujuan gambar di dalam folder kategori
    destination_path = os.path.join(folder_path, image_name)
    
    # Cek apakah file gambar sudah ada sebelum menyalin
    if not os.path.exists(destination_path):
        shutil.copy(image_path, destination_path)
        print(f"✅ Gambar {image_name} disimpan ke kategori: {category}")
    else:
        print(f"✅ Gambar {image_name} sudah ada di kategori {category}, tidak perlu disalin ulang.")


In [ ]:
# Path ke folder testing
TESTING_FOLDER = "/kaggle/input/testing"

# Ambil semua file gambar dalam folder testing
image_files = [f for f in os.listdir(TESTING_FOLDER) if f.endswith((".jpg", ".jpeg", ".png"))]

if not image_files:
    raise FileNotFoundError("❌ Tidak ada gambar yang ditemukan di /kaggle/input/testing")

print(f"✅ {len(image_files)} gambar ditemukan di {TESTING_FOLDER}")


In [ ]:
# Loop untuk memproses setiap gambar
for image_file in image_files:
    IMAGE_PATH = os.path.join(TESTING_FOLDER, image_file)

    try:
        # Ekstrak fitur dari gambar langsung dari folder /kaggle/input/testing/
        feature = extract_features(IMAGE_PATH)

        # Prediksi caption untuk gambar
        predicted_caption = predict_caption(model, feature, tokenizer, max_length)

        # Tentukan kategori berdasarkan caption yang diprediksi
        category = categorize_image_by_keywords(predicted_caption)

        # Simpan gambar ke dalam folder kategori (masih di /kaggle/input/testing/)
        save_image_to_category_folder(image_file, IMAGE_PATH, category)

        # Tampilkan gambar dengan caption
        image = Image.open(IMAGE_PATH)
        plt.imshow(image)
        plt.axis("off")
        plt.title(f"Predicted Caption: {predicted_caption}\nKategori: {category}", fontsize=14)
        plt.show()

        # Print hasil
        print(f"\n📝 **Predicted Caption:** {predicted_caption}")
        print(f"📂 **Kategori Gambar:** {category}")

    except Exception as e:
        print(f"❌ Error saat memproses {image_file}: {e}")

In [ ]:
import random
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# Pastikan jumlah sampel tidak lebih dari jumlah gambar yang tersedia
num_samples = min(1000, len(image_files))

# Ambil 1000 gambar secara acak dari dataset Flickr8k
sampled_images = random.sample(image_files, num_samples)

# Smoothing function untuk BLEU score
smooth = SmoothingFunction().method1

# Simpan skor BLEU untuk setiap gambar
bleu1_scores = []

# Proses setiap gambar yang dipilih
for image_name in sampled_images:
    image_id = os.path.splitext(image_name)[0]  
    
    # Ambil caption asli dari dataset Flickr8k
    actual_captions = mapping.get(image_id, ["Tidak ada caption asli"])

    # Prediksi caption
    predicted_caption = predict_caption(model, features[image_id], tokenizer, max_length)

    # Hitung BLEU-1 Score
    bleu1 = calculate_bleu_score(actual_captions, predicted_caption)

    # Simpan hasil skor BLEU-1
    bleu1_scores.append(bleu1)

# Hitung rata-rata BLEU-1 Score
average_bleu1 = sum(bleu1_scores) / len(bleu1_scores)

# Tampilkan hasil rata-rata
average_bleu1
